In [ ]:
# Cell 1 — install, then force restart
!pip install -U cellxgene_census -q

In [2]:
import os, glob
import cellxgene_census
import scanpy as sc
import pandas as pd

CENSUS_VERSION = "2025-11-08"
DATASET_ID = "0b75c598-0893-4216-afe8-5414cab7739d"
OUTPUT_DIR = "/content/kpmp_donor_pseudobulk"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(cellxgene_census.__version__)
print(DATASET_ID, OUTPUT_DIR)

1.18.0
0b75c598-0893-4216-afe8-5414cab7739d /content/kpmp_donor_pseudobulk


# Data pull

In [3]:
# Pipeline

def get_donor_disease(dataset_id, census_version=CENSUS_VERSION):
    """Fetch one row per donor with their disease label."""
    with cellxgene_census.open_soma(census_version=census_version) as census:
        obs = cellxgene_census.get_obs(
            census, organism="Homo sapiens",
            value_filter=f"dataset_id == '{dataset_id}' and disease in ['chronic kidney disease', 'normal']",
        )
    return obs.drop_duplicates("donor_id")[["donor_id", "disease"]]


def run_pseudobulk(donor_disease, dataset_id, output_dir=OUTPUT_DIR, census_version=CENSUS_VERSION):
    """Per-donor mean expression across all genes (normalized + log1p), skipping donors already saved."""
    for _, (donor_id, disease) in donor_disease[["donor_id", "disease"]].iterrows():
        out_path = f"{output_dir}/{donor_id}.csv"
        if os.path.exists(out_path):
            continue

        with cellxgene_census.open_soma(census_version=census_version) as census:
            adata = cellxgene_census.get_anndata(
                census, organism="Homo sapiens",
                obs_value_filter=f"dataset_id == '{dataset_id}' and donor_id == '{donor_id}'",
            )
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        mean_expr = pd.Series(adata.X.toarray().mean(axis=0),
                               index=adata.var["feature_name"].values, name=donor_id)
        mean_expr.to_csv(out_path, header=True)
        print(f"Saved {donor_id} ({disease}) — {adata.shape[0]} cells")


def build_matrix(donor_disease, output_dir=OUTPUT_DIR):
    """Merge per-donor CSVs, dedupe genes, attach disease labels, return (genes x donors, donors x genes)."""
    files = glob.glob(f"{output_dir}/*.csv")
    matrix = pd.concat([pd.read_csv(f, index_col=0) for f in files], axis=1)
    matrix = matrix.groupby(matrix.index).mean()  # collapse any duplicate gene rows

    matrix_T = matrix.T
    matrix_T["disease"] = matrix_T.index.map(donor_disease.set_index("donor_id")["disease"])

    print(f"{len(files)} donors -> matrix {matrix.shape}, "
          f"dup genes: {matrix.index.duplicated().sum()}, NaNs: {matrix.isna().sum().sum()}")
    print(matrix_T["disease"].value_counts())
    return matrix, matrix_T

In [4]:
# Run
donor_disease = get_donor_disease(DATASET_ID)
print(f"Total donors: {len(donor_disease)}")
print(donor_disease["disease"].value_counts())

run_pseudobulk(donor_disease, DATASET_ID)

donor_matrix, donor_matrix_T = build_matrix(donor_disease)
donor_matrix.to_csv("/content/kpmp_all_donors_genes_by_donor_CLEAN.csv")
donor_matrix_T.to_csv("/content/kpmp_all_donors_donor_by_gene_with_disease_CLEAN.csv")

Total donors: 50
disease
normal                                                 31
chronic kidney disease                                 19
lymphoma                                                0
lymphangioleiomyomatosis                                0
lung large cell carcinoma                               0
                                                       ..
dementia || Alzheimer disease || Lewy body dementia     0
dementia || Alzheimer disease                           0
dementia                                                0
cytomegalovirus infection                               0
dementia || Alzheimer disease || diabetes mellitus      0
Name: count, Length: 259, dtype: int64


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Saved 31-10001 (chronic kidney disease) — 6999 cells


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Saved 31-10000 (chronic kidney disease) — 10066 cells


...

Saved Sample1158-EO3 (normal) — 958 cells


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Saved Sample1162-EO1 (normal) — 962 cells


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Saved Sample1162-EO2 (normal) — 1329 cells
50 donors -> matrix (60028, 50), dup genes: 0, NaNs: 0
disease
normal                                                 31
chronic kidney disease                                 19
lymphoma                                                0
lymphangioleiomyomatosis                                0
lung large cell carcinoma                               0
                                                       ..
dementia || Alzheimer disease || Lewy body dementia     0
dementia || Alzheimer disease                           0
dementia                                                0
cytomegalovirus infection                               0
dementia || Alzheimer disease || diabetes mellitus      0
Name: count, Length: 259, dtype: int64


In [5]:
# Save a label manifest so disease status travels with the download
donor_disease.to_csv(f"{OUTPUT_DIR}/_sample_labels.csv", index=False)
print(donor_disease["disease"].value_counts())

disease
normal                                                 31
chronic kidney disease                                 19
lymphoma                                                0
lymphangioleiomyomatosis                                0
lung large cell carcinoma                               0
                                                       ..
dementia || Alzheimer disease || Lewy body dementia     0
dementia || Alzheimer disease                           0
dementia                                                0
cytomegalovirus infection                               0
dementia || Alzheimer disease || diabetes mellitus      0
Name: count, Length: 259, dtype: int64


In [6]:
import shutil
from google.colab import files

shutil.make_archive("/content/kpmp_donor_pseudobulk", "zip", OUTPUT_DIR)
files.download("/content/kpmp_donor_pseudobulk.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>